In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage # by default your LLM try --> input --> human or AI meeage
from langchain_core.chat_history import BaseChatMessageHistory
from pydantic import BaseModel, Field
from typing import List

In [2]:
load_dotenv()

True

In [18]:

class WindowChatMessageHistory(BaseChatMessageHistory, BaseModel):
    """
    A chat message history that only keeps the last `k` messages.
    
    When the history exceeds k messages, the OLDEST messages
    are silently dropped. This creates a sliding window effect.
    
    Implements BaseChatMessageHistory interface so it works
    seamlessly with RunnableWithMessageHistory.
    """
    messages: List[BaseMessage] = Field(default_factory=list)
    k: int = Field(default=6)  # keep last k messages (= k/2 turns)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """
        Add new messages, then trim to last k.
        Called automatically by RunnableWithMessageHistory.
        """
        self.messages.extend(messages)
        
        # Trim: keep only the last k messages
        # k=6 means 3 HumanMessages + 3 AIMessages = 3 turns
        if len(self.messages) > self.k:
            dropped = len(self.messages) - self.k
            self.messages = self.messages[-self.k:]
            print(f"  [Window] Dropped {dropped} oldest message(s). Now: {len(self.messages)} messages.")

    def clear(self) -> None:
        """Clear all messages."""
        self.messages = []







In [19]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [32]:

WINDOW_K = 8  # keep last 4 messages = 2 full turns (human + AI each)

prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify: Billing / Technical / General. Priority: High / Medium / Low.
Remember customer details within the conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt | llm | StrOutputParser()

In [33]:
window_store = {}

def get_session_history(session_id):
    if session_id not in window_store:
        window_store[session_id] = WindowChatMessageHistory(k=WINDOW_K)
    return window_store[session_id]

In [34]:
window_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

/Users/rahultiwari/Documents/02_Freelancing/Hachion_batch/ai_engineering_19th_may/ai-env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [35]:
cfg = {"configurable": {"session_id": "window_john"}}
window_assistant.invoke({"input":"Billing complaint from John — invoice #1042, overcharged $250."},
                        config=cfg)



'Classification: Billing  \nPriority: High  \n\nCustomer details: John; Invoice #1042; Overcharged $250.'

In [36]:
window_store

{'window_john': WindowChatMessageHistory(messages=[HumanMessage(content='Billing complaint from John — invoice #1042, overcharged $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Classification: Billing  \nPriority: High  \n\nCustomer details: John; Invoice #1042; Overcharged $250.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], k=8)}

In [37]:
turns = [
    "Billing complaint from John — invoice #1042, overcharged $250.",
    "What category is his complaint?",
    "What is the SLA for billing?",         # ← Turn 3 — Turn 1 gets dropped after this
    "Can you remind me who made the original complaint?",  # ← CRITICAL: Turn 1 is now GONE
    "Draft a 2-line apology email for him.",
]

In [38]:
for i, turn in enumerate(turns, 1):
    response = window_assistant.invoke({"input": turn}, config=cfg)
    hist = window_store["window_john"]
    print(response)

Classification: Billing  
Priority: High  

Customer details: John; Invoice #1042; Overcharged $250.
John's complaint falls under the category of Billing.
The typical Service Level Agreement (SLA) for billing inquiries is usually 24 to 48 hours for an initial response. However, resolution times may vary depending on the complexity of the issue. Please confirm the specific SLA details with your organization for accuracy.
  [Window] Dropped 2 oldest message(s). Now: 8 messages.
The original complaint was made by John regarding invoice #1042, where he was overcharged by $250.
  [Window] Dropped 2 oldest message(s). Now: 8 messages.
Subject: Apology for Billing Discrepancy  

Dear John,  

We sincerely apologize for the inconvenience caused by the overcharge on invoice #1042. We are currently investigating the issue and will resolve it promptly.  

Best regards,  
[Your Name]  
[Your Company]  


In [39]:
window_store['window_john'].messages

[HumanMessage(content='What category is his complaint?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the SLA for billing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The typical Service Level Agreement (SLA) for billing inquiries is usually 24 to 48 hours for an initial response. However, resolution times may vary depending on the complexity of the issue. Please confirm the specific SLA details with your organization for accuracy.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you remind me who made the original complaint?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The original complaint was made by John regarding invoice #1042, where he was overcharged by $250.', additional_kwargs={}, re

In [6]:
message = ['abc','xyz','abc','I am ok']

updatred_list = []
k = 3

if len(message) > k:
    updatred_list = message[k:]
else:
    updatred_list = message

updatred_list

['I am ok']